# SentinelMail — BERT Fine-Tuned: Training & Evaluation

Fine-tunes `bert-base-uncased` on the ai4privacy training split and evaluates it on the
validation split. Saves metrics to `evaluation/results/bert_metrics.json` for ensemble comparison.

**Prerequisites**
- `pip install transformers torch`
- HuggingFace model weights downloaded automatically on first run

## 0. Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from tqdm.auto import tqdm

REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT))

CHECKPOINT_DIR = REPO_ROOT / "models" / "bert" / "checkpoint"
RESULTS_DIR    = REPO_ROOT / "evaluation" / "results"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
from data.preprocessing import load_split, LABEL_COLS
from models.bert.dataset import DLPDataset, collate_fn
from models.bert.model   import BertForDLPClassification
from models.bert.predict import BertDLPDetector
from evaluation import (
    compute_all_metrics,
    classify_errors,
    error_summary,
    plot_confusion_matrices,
    plot_per_label_bars,
    plot_roc_curves,
    plot_pr_curves,
    save_results,
    from_dataframe,
    LABEL_COLS,
)

## 1. Load Data

In [ ]:
print("Loading splits...")
train_df = load_split("train")
val_df   = load_split("validation")

train_texts  = train_df["source_text"].tolist()
val_texts    = val_df["source_text"].tolist()
train_labels = train_df[LABEL_COLS].values.astype(np.float32)
val_labels   = val_df[LABEL_COLS].values.astype(np.float32)

print(f"Train: {len(train_texts):,} samples")
print(f"Val:   {len(val_texts):,} samples")
print()
print("Label distribution (train):")
print(train_df[LABEL_COLS].sum().to_string())

## 2. Encode Inputs

In [ ]:
MODEL_NAME = "bert-base-uncased"
BATCH_SIZE = 16

print(f"Loading {MODEL_NAME} encoder...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_ds = DLPDataset(train_texts, train_labels, tokenizer)
val_ds   = DLPDataset(val_texts,   val_labels,   tokenizer)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate_fn, num_workers=2, pin_memory=(DEVICE.type == "cuda"),
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn,
)

print(f"Train batches: {len(train_loader):,}  |  Val batches: {len(val_loader):,}")

## 3. Initialise Model

In [ ]:
model = BertForDLPClassification().to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params:     {total_params:,}")
print(f"Trainable params: {trainable_params:,}  (blocks 10-11 + head)")

## 4. Train

Skip this cell if `checkpoint/best_model.pt` already exists — section 5 loads it.

Early stopping on **val macro-F1** (not val loss) as per ADR-001.

In [ ]:
EPOCHS           = 5
LR               = 2e-5
LABEL_SMOOTHING  = 0.05
WARMUP_FRACTION  = 0.10
CKPT             = CHECKPOINT_DIR / "best_model.pt"


def _pos_weights(labels: np.ndarray, device: torch.device) -> torch.Tensor:
    pos = labels.sum(axis=0).clip(1)
    neg = len(labels) - pos
    return torch.tensor(neg / pos, dtype=torch.float32).to(device)


def _smooth(labels: torch.Tensor, eps: float = LABEL_SMOOTHING) -> torch.Tensor:
    return labels * (1.0 - eps) + (1.0 - labels) * eps


def _macro_f1_from_logits(logits_list, labels_list) -> float:
    proba  = torch.sigmoid(torch.cat(logits_list)).cpu().numpy()
    y_pred = (proba >= 0.5).astype(int)
    y_true = torch.cat(labels_list).cpu().numpy().astype(int)
    f1s = []
    for i in range(4):
        tp = ((y_pred[:, i] == 1) & (y_true[:, i] == 1)).sum()
        fp = ((y_pred[:, i] == 1) & (y_true[:, i] == 0)).sum()
        fn = ((y_pred[:, i] == 0) & (y_true[:, i] == 1)).sum()
        p  = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        r  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1s.append(2 * p * r / (p + r) if (p + r) > 0 else 0.0)
    return float(np.mean(f1s))


criterion = nn.BCEWithLogitsLoss(pos_weight=_pos_weights(train_labels, DEVICE))
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=0.01
)
total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(WARMUP_FRACTION * total_steps)
scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

history = {"train_loss": [], "val_loss": [], "val_f1": []}
best_val_f1 = 0.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for input_ids, attention_mask, labels in tqdm(train_loader, desc=f"Epoch {epoch:02d} train", leave=False):
        input_ids, attention_mask, labels = (
            input_ids.to(DEVICE), attention_mask.to(DEVICE), labels.to(DEVICE)
        )
        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss   = criterion(logits, _smooth(labels))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        train_loss += loss.item()

    model.eval()
    val_loss = 0.0
    val_logits_all, val_labels_all = [], []
    with torch.no_grad():
        for input_ids, attention_mask, labels in val_loader:
            input_ids, attention_mask, labels_dev = (
                input_ids.to(DEVICE), attention_mask.to(DEVICE), labels.to(DEVICE)
            )
            logits    = model(input_ids, attention_mask)
            val_loss += criterion(logits, _smooth(labels_dev)).item()
            val_logits_all.append(logits.detach())
            val_labels_all.append(labels)

    avg_train = train_loss / len(train_loader)
    avg_val   = val_loss   / len(val_loader)
    val_f1    = _macro_f1_from_logits(val_logits_all, val_labels_all)

    history["train_loss"].append(avg_train)
    history["val_loss"].append(avg_val)
    history["val_f1"].append(val_f1)

    marker = "  ✓ saved" if val_f1 > best_val_f1 else ""
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), CKPT)
    print(f"Epoch {epoch:02d}/{EPOCHS}  train={avg_train:.4f}  val={avg_val:.4f}  macro-F1={val_f1:.4f}{marker}")

print(f"\nBest val macro-F1: {best_val_f1:.4f}  →  {CKPT}")

### 4a. Learning Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epochs_range = range(1, len(history["train_loss"]) + 1)

axes[0].plot(epochs_range, history["train_loss"], label="train")
axes[0].plot(epochs_range, history["val_loss"],   label="val")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("BCE Loss")
axes[0].set_title("BERT — Learning Curves (Loss)")
axes[0].legend()

axes[1].plot(epochs_range, history["val_f1"], color="green", marker="o")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Macro-F1")
axes[1].set_title("BERT — Val Macro-F1")

plt.tight_layout()
plt.show()

## 5. Load Best Checkpoint

In [ ]:
CKPT = CHECKPOINT_DIR / "best_model.pt"

detector = BertDLPDetector(
    checkpoint_path=str(CKPT),
    device=str(DEVICE),
)
print(f"Loaded: {CKPT.name}")

## 6. Run Inference on Validation Set

In [ ]:
THRESHOLD = 0.5

print(f"Running predict_proba on {len(val_texts):,} samples...")
y_proba = detector.predict_proba(val_texts)       # (N, 4) float32
y_pred  = (y_proba >= THRESHOLD).astype(np.int32) # (N, 4) int
y_true  = from_dataframe(val_df)                   # (N, 4) int

print(f"y_proba range: [{y_proba.min():.3f}, {y_proba.max():.3f}]")
print()
print(f"{'label':14s}  {'pred':>6s}  {'true':>6s}")
for i, col in enumerate(LABEL_COLS):
    print(f"  {col:12s}  {y_pred[:,i].sum():6d}  {y_true[:,i].sum():6d}")

## 7. Metrics

In [ ]:
metrics = compute_all_metrics(y_true, y_pred, y_proba=y_proba)

ci = metrics["macro_f1_ci"]
print(f"Macro-F1:      {metrics['macro_f1']:.4f}  "
      f"(95% CI: [{ci['lower']:.4f}, {ci['upper']:.4f}])")
print(f"Macro AUC-ROC: {metrics['auc_roc']['macro']:.4f}")
print()

pd.DataFrame(metrics["per_label"]).T.round(4)

### 7a. Threshold Sensitivity (0.3 / 0.5 / 0.7)

In [ ]:
rows = []
for t in [0.3, 0.5, 0.7]:
    m   = compute_all_metrics(y_true, (y_proba >= t).astype(np.int32), n_bootstrap=50)
    row = {"threshold": t, "macro_f1": round(m["macro_f1"], 4)}
    for label in LABEL_COLS:
        row[f"{label}_f1"] = round(m["per_label"][label]["f1"], 4)
    rows.append(row)

pd.DataFrame(rows).set_index("threshold")

## 8. Confusion Matrices & Per-Label Bars

In [ ]:
_, fig = plot_confusion_matrices(y_true, y_pred, normalize="true", return_fig=True)
fig.suptitle("BERT — Confusion Matrices (val EN, threshold=0.5)", y=1.02)
plt.show()

_, bar_fig = plot_per_label_bars(metrics, return_fig=True)
bar_fig.suptitle("BERT — Per-Label Metrics")
plt.show()

## 9. ROC & PR Curves

In [ ]:
roc_data = plot_roc_curves(y_true, y_proba)
pr_data  = plot_pr_curves(y_true, y_proba)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
COLORS = ["steelblue", "darkorange", "green", "red"]

for i, label in enumerate(LABEL_COLS):
    axes[0].plot(roc_data[label]["fpr"], roc_data[label]["tpr"],
                 label=f"{label} (AUC={roc_data[label]['auc']:.3f})", color=COLORS[i])
    axes[1].plot(pr_data[label]["recall"], pr_data[label]["precision"],
                 label=f"{label} (AP={pr_data[label]['ap']:.3f})", color=COLORS[i])

axes[0].plot([0, 1], [0, 1], "k--", alpha=0.4)
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")
axes[0].set_title("ROC Curves"); axes[0].legend(fontsize=9)
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curves"); axes[1].legend(fontsize=9)
fig.suptitle("BERT — Validation Set")
plt.tight_layout()
plt.show()

## 10. Error Analysis

Classifies FP/FN into the four taxonomy buckets (descriptive error analysis for RQ1/RQ2 model comparison).

In [ ]:
texts = val_df["source_text"].tolist()

error_results = classify_errors(
    texts, y_true, y_pred,
    max_examples=20,
)

summary_df = error_summary(error_results)
print("Error counts by type and label:")
display(summary_df.pivot(index="error_type", columns="label", values="count").fillna(0).astype(int))

In [ ]:
def show_bucket(key: str, n: int = 5) -> None:
    records = error_results.get(key, [])
    print(f"\n{'─'*60}")
    print(f"{key}  ({len(records)} total, showing {min(n, len(records))})")
    print(f"{'─'*60}")
    for r in records[:n]:
        print(f"  [{r['label']}] {r['text_snippet'][:120].replace(chr(10), ' ')}")

show_bucket("FP_negation")
show_bucket("FP_hypothetical")
show_bucket("FN_obfuscated")
show_bucket("FN_implicit")

## 11. Save Results

In [ ]:
out_path = RESULTS_DIR / "bert_metrics.json"

save_results(
    metrics,
    path=out_path,
    model_name="bert_finetuned",
    n_samples=len(val_df),
    threshold=THRESHOLD,
    extra_metadata={
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "lr": LR,
        "base_model": MODEL_NAME,
        "frozen_blocks": "0-9",
        "label_smoothing": LABEL_SMOOTHING,
        "checkpoint": str(CKPT),
    },
)

print(f"Saved to {out_path}")